# Admin setup

**Who runs this notebook:** workspace **admins** (or the person who imports the workshop once).

**Who does not:** workshop participants. Participants start at `00_start_here`.

Use this notebook **after** you finish the written checklist ([ADMIN_CHECKLIST.md](https://github.com/archana986/ai-literacy-workshop/blob/main/ADMIN_CHECKLIST.md)). Run All here to prove the room is ready before class.

## Before you run (outside this notebook)

Complete these once (details in **ADMIN_CHECKLIST.md**):

1. Catalog with participant `USE CATALOG` + `CREATE SCHEMA`
2. Model access: prefer UC model service `EXECUTE` (system.ai); fallback legacy endpoint `CAN QUERY`
3. SQL warehouse with participant `CAN USE` (Genie optional)
4. Optional PII account group for the presenter demo
5. Import notebooks + `assets/*.png` as workspace **FILE** objects next to the notebooks


## Step 0. Set your lab values (widget bar)

Run the next cell. A **widget bar** appears at the top of this notebook. Fill in your values there, then **Run All**.

Values you confirm are saved to your home folder, so later labs open with the same values. You can still change them in any lab's widget bar.

In [ ]:
# Creates the widget bar for THIS notebook. Edit values in the bar at the top, not in code.
import json, os
_LAB_USER = spark.sql("SELECT current_user()").first()[0]
_LAB_SETTINGS = f"/Workspace/Users/{_LAB_USER}/ai_literacy_lab_settings.json"
try:
    with open(_LAB_SETTINGS) as _f:
        _saved = json.load(_f)
except Exception:
    _saved = {}
for _name, _default, _label in [
    ("catalog", "main", "1. Catalog"),
    ("schema", "", "2. Schema (blank = auto from your login)"),
    ("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "3. LLM model (UC FQN or legacy endpoint)"),
    ("warehouse_name", "Serverless Starter Warehouse", "4. SQL warehouse"),
    ("demo_filter_state", "CA", "5. State hidden by row filter"),
    ("pii_group", "lab_pii_viewers", "6. Group that sees unmasked PII"),
]:
    dbutils.widgets.text(_name, _saved.get(_name, _default), _label)
print("Widget bar ready at the top of this notebook." + (" Loaded your saved values." if _saved else " Using defaults; set yours, then Run All."))

## Install packages

Needed for endpoint probes and diagram helpers.

In [ ]:
%pip install -U -q "mlflow[databricks]>=3.4" "databricks-sdk>=0.60" openai httpx
dbutils.library.restartPython()

## Load config

Reads the values from the **Step 0 widget bar** at the top of this notebook. Set **catalog**, **schema** (a throwaway admin schema, or blank for auto), **llm_endpoint**, **warehouse_name**, and **pii_group** to the values participants will use. Changed a widget? Re-run from this cell.

In [ ]:
%run ./_config

## Check diagram assets are FILEs

Lab notebooks call `show_diagram(...)`. PNGs must live under `./assets/` as workspace **FILE** objects (not notebooks).

In [ ]:
from pathlib import Path
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ObjectType

w = WorkspaceClient()
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
assets_ws = f"{Path(nb_path).parent}/assets"
print("Looking under:", assets_ws)

required = [
    "what_we_build.png",
    "lab1_journey.png",
    "lab1_governance.png",
    "lab1_tools.png",
    "lab2_playground.png",
    "lab2_agent_loop.png",
    "ideation_canvas.png",
]
missing, wrong_type = [], []
for name in required:
    path = f"{assets_ws}/{name}"
    try:
        st = w.workspace.get_status(path)
        ot = st.object_type
        print(f"  {name:28}  {ot}")
        if ot != ObjectType.FILE:
            wrong_type.append(name)
    except Exception as e:
        print(f"  {name:28}  MISSING ({e})")
        missing.append(name)

if missing or wrong_type:
    raise AssertionError(
        "Fix assets before class. Missing: "
        + ", ".join(missing or ["(none)"])
        + "; not FILE: "
        + ", ".join(wrong_type or ["(none)"])
        + ". Re-import PNGs as FILE into the assets/ folder."
    )
print("\nAll required diagram assets found as FILE.")

## Prove diagrams render

If this cell shows the runtime-flow image, participants will see lab diagrams too.

In [ ]:
show_diagram("what_we_build.png", "Admin smoke: diagram assets load")

## Resolved values

In [ ]:
print(f"""
  You are ............ {USER}
  Workshop catalog ... {CATALOG}
  Smoke schema ....... {FQ_PLAIN}
  LLM endpoint ....... {LLM_ENDPOINT}
  SQL warehouse ...... {WAREHOUSE_NAME}
  Hidden state ....... {HIDDEN_STATE}
  PII group .......... {PII_GROUP}
""")

## Environment preflight

Same probes participants will hit in `00_start_here`. **FAIL** means fix grants or widgets before class. PII group may WARN if you are not in the presenter group (OK).

### Check helpers

Small helpers used by the checks below.

In [ ]:
from databricks.sdk import WorkspaceClient
import mlflow

w = WorkspaceClient()
results = []  # (name, status, detail, required)


def check(name, fn, required=True):
    """Run one probe; store PASS/FAIL/WARN without stopping the loop."""
    try:
        detail = fn()
        results.append((name, "PASS", detail or "", required))
    except Exception as e:
        results.append((name, "FAIL" if required else "WARN", str(e)[:220], required))

### Catalog and schema probes

In [ ]:
def catalog_access():
    try:
        n = len(spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`").collect())
        return f"{n} schemas visible in `{CATALOG}`"
    except Exception as e:
        raise RuntimeError(
            f"Cannot USE CATALOG `{CATALOG}`. Ask the instructor to grant USE CATALOG "
            f"(and CREATE SCHEMA). ({e})"
        ) from e


def create_schema():
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ} COMMENT 'AI Literacy lab schema for {USER}'")
        spark.sql(f"COMMENT ON SCHEMA {FQ} IS 'AI Literacy lab schema for {USER}'")
        return f"{FQ_PLAIN} ready"
    except Exception as e:
        raise RuntimeError(
            f"CREATE SCHEMA denied on `{CATALOG}`. Ask for GRANT CREATE SCHEMA. ({e})"
        ) from e


def create_function():
    # Lab 1 needs CREATE FUNCTION for tools, row filter, and mask
    spark.sql(f"CREATE OR REPLACE FUNCTION {FQ}._lab_preflight() RETURNS INT RETURN 1")
    spark.sql(f"DROP FUNCTION IF EXISTS {FQ}._lab_preflight")
    return "CREATE FUNCTION works in your schema"

check("Catalog access (USE CATALOG)", catalog_access)
check("Can create your schema (CREATE SCHEMA)", create_schema)
check("Can create functions in your schema", create_function)

### Endpoint, warehouse, and MLflow probes

In [ ]:
def llm_endpoint():
    ok, detail, mode = probe_llm_access()
    if not ok:
        raise RuntimeError(detail)
    return detail


def sql_warehouse():
    x = resolve_sql_warehouse()
    return f"{x.name} id={x.id}"


def mlflow_home():
    mlflow.set_experiment(EXPERIMENT_PATH)
    return EXPERIMENT_PATH


check("LLM model reachable (UC Gateway or legacy endpoint)", llm_endpoint)
check("SQL warehouse visible (CAN USE)", sql_warehouse)
check("MLflow experiment in your home folder", mlflow_home)


### PII group probe (WARN only)

Attendees should usually **not** be members. The notebook never creates the group.

In [ ]:
def pii_group():
    member = bool(spark.sql(f"SELECT is_account_group_member('{PII_GROUP}')").first()[0])
    role = (
        "you ARE a member: you will see real names and hidden-state rows (presenter view)"
        if member else
        "you are NOT a member: names masked, hidden state filtered (expected attendee view)"
    )
    try:
        found = any(True for _ in w.groups.list(filter=f'displayName eq "{PII_GROUP}"'))
    except Exception as e:
        raise RuntimeError(
            f"Could not list account groups ({type(e).__name__}). Instructor must create "
            f"'{PII_GROUP}' and add only the presenter. {role}"
        ) from e
    if not found:
        raise RuntimeError(
            f"Account group '{PII_GROUP}' was not found. Admin creates it (presenter only). {role}"
        )
    return f"group '{PII_GROUP}' exists. {role}"

check(f"Account group '{PII_GROUP}' (instructor creates)", pii_group, required=False)

### Preflight summary

In [ ]:
for name, status, detail, _ in results:
    print(f"{status:5}  {name}\n       {detail}")

hard_fail = [n for n, s, _, req in results if req and s != "PASS"]
if hard_fail:
    raise AssertionError("Fix FAIL items before unlocking the room: " + "; ".join(hard_fail))

print("\nAdmin setup PASS. Participants can use 00_start_here.")
print("Optional next: Run All on 01_setup_governed_data once in this schema to prove Lab 1.")
print("If the PII group check is WARN and you are not the presenter, that is expected.")

## Next (admin)

1. If everything above is green, the room is ready.
2. Optional: open `01_setup_governed_data` and Run All once yourself.
3. Point participants to **`00_start_here`** only. They do not need this admin notebook.